# DKT Atlas Surface-Area Linear Regression

This notebook is a separate first-pass model for predicting `p_factor`. It leaves the original `analysis.ipynb` untouched.

Model idea:
- use FreeSurfer rows where `atlas == 'aparc.DKTatlas'`
- use each region's `SurfArea` as a predictor
- fit a linear regression model
- evaluate with cross-validated predictions on the training participants
- fit once on all training participants and predict the held-out test participants

## Imports and Settings

In [1]:
from pathlib import Path
from urllib.error import HTTPError

import numpy as np
import pandas as pd
from rbclib import RBCPath

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ATLAS = "aparc.DKTatlas"
MEASURE = "SurfArea"

RBC_DATA_PATH = Path("/home/jovyan/shared/data/RBC")
TRAIN_FILE = RBC_DATA_PATH / "train_participants.tsv"
TEST_FILE = RBC_DATA_PATH / "test_participants.tsv"

# The feature table is saved locally after the first extraction so later runs
# do not need to re-download every participant's FreeSurfer table.
FEATURE_CACHE = Path("cache/dkt_surfarea_features.tsv")
FS_CACHE_DIR = Path.home() / "cache" / "rbc_freesurfer"

OUTPUT_PATH = Path("results/dkt_surfarea_linear.tsv")

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Load Participant Metadata

In [2]:
if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        "Expected train/test TSVs under /home/jovyan/shared/data/RBC. "
        "Run this notebook on the NeuroHackademy JupyterHub."
    )

train_data = pd.read_csv(TRAIN_FILE, sep="\t")
test_data = pd.read_csv(TEST_FILE, sep="\t")
all_data = pd.concat([train_data, test_data], ignore_index=True)

print(f"Training participants: {len(train_data)}")
print(f"Test participants: {len(test_data)}")
all_data.head()

Training participants: 1280
Test participants: 321


,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,p_factor
0,1000393599,PNC,PNC1,PNC1,1,15.583333,Male,Black,not Hispanic or Latino,22.15,Right,9th Grade,Complete primary,Complete secondary,0.589907
1,1000881804,PNC,PNC1,PNC1,1,14.916667,Male,Black,not Hispanic or Latino,21.52,Right,7th Grade,Complete secondary,Complete secondary,-0.655377
2,1001970838,PNC,PNC1,PNC1,1,17.833333,Male,Other,Hispanic or Latino,23.98,Right,11th Grade,Complete tertiary,Complete tertiary,-0.659061
3,100527940,PNC,PNC1,PNC1,1,8.250000,Male,Black,not Hispanic or Latino,NaN,Ambidextrous,1st Grade,Complete secondary,Complete primary,-0.591516
4,1006151876,PNC,PNC1,PNC1,1,21.500000,Female,Other,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,-0.377828


## Extract DKT `SurfArea` Features

Each DKT region becomes one column. If the same `StructName` appears more than once, those surface areas are summed for that participant.

In [3]:
fs_root = RBCPath(
    "rbc://PNC_FreeSurfer/freesurfer",
    local_cache_dir=FS_CACHE_DIR,
)


def participant_label(participant_id):
    """Return the participant id text used in FreeSurfer filenames."""
    text = str(participant_id)
    if text.startswith("sub-"):
        text = text[4:]
    if text.endswith(".0"):
        text = text[:-2]
    return text


def load_regionsurfacestats(participant_id):
    """Load one participant's FreeSurfer region surface stats table."""
    pid = participant_label(participant_id)
    tsv_path = fs_root / f"sub-{pid}" / f"sub-{pid}_regionsurfacestats.tsv"
    with tsv_path.open("r") as f:
        return pd.read_csv(f, sep="\t")


def extract_dkt_surfareas(participant_id):
    """Extract DKT atlas SurfArea values as a named feature vector."""
    data = load_regionsurfacestats(participant_id)
    dkt = data.loc[data["atlas"].eq(ATLAS), ["StructName", MEASURE]].copy()

    if dkt.empty:
        raise ValueError(f"No rows found for atlas={ATLAS!r}, participant={participant_id}")

    dkt[MEASURE] = pd.to_numeric(dkt[MEASURE], errors="coerce")
    features = dkt.groupby("StructName")[MEASURE].sum(min_count=1)
    features.index = [f"{MEASURE}__{name}" for name in features.index]
    return features


def build_dkt_feature_table(participant_ids, feature_cache=FEATURE_CACHE):
    """Build or load the participant-by-region DKT SurfArea feature table."""
    if feature_cache.exists():
        print(f"Loading cached features from {feature_cache}")
        return pd.read_csv(feature_cache, sep="\t")

    feature_cache.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    total = len(participant_ids)

    for ii, participant_id in enumerate(participant_ids, start=1):
        if ii == 1 or ii % 25 == 0 or ii == total:
            print(f"Extracting {ii}/{total}: participant {participant_id}")

        row = {"participant_id": participant_id}
        try:
            row.update(extract_dkt_surfareas(participant_id).to_dict())
        except (FileNotFoundError, HTTPError, ValueError) as err:
            print(f"  Missing/unusable FreeSurfer data for {participant_id}: {err}")
        rows.append(row)

    feature_table = pd.DataFrame(rows)
    feature_table.to_csv(feature_cache, sep="\t", index=False)
    print(f"Saved feature cache to {feature_cache}")
    return feature_table

In [4]:
feature_data = build_dkt_feature_table(all_data["participant_id"].tolist())

model_data = all_data[["participant_id", "p_factor"]].merge(
    feature_data,
    on="participant_id",
    how="left",
)

feature_cols = [col for col in model_data.columns if col.startswith(f"{MEASURE}__")]
train_rows = model_data["p_factor"].notna()

# Keep only feature columns that are observed for at least one training participant.
usable_feature_cols = [
    col for col in feature_cols
    if model_data.loc[train_rows, col].notna().any()
]

print(f"Extracted DKT SurfArea predictors: {len(feature_cols)}")
print(f"Usable predictors in training data: {len(usable_feature_cols)}")
model_data.loc[:, ["participant_id", "p_factor"] + usable_feature_cols[:5]].head()

Extracting 1/1601: participant 1000393599
Extracting 25/1601: participant 1060532313
Extracting 50/1601: participant 1139941451
Extracting 75/1601: participant 1207361199
Extracting 100/1601: participant 1313957126
  Missing/unusable FreeSurfer data for 1342487188: [Errno 2] No such file or directory: '/home/jovyan/shared/data/RBC/repos/PNC_FreeSurfer/freesurfer/sub-1342487188/sub-1342487188_regionsurfacestats.tsv'
Extracting 125/1601: participant 1402618672
Extracting 150/1601: participant 1484543503
Extracting 175/1601: participant 1547173705
Extracting 200/1601: participant 1620557808
Extracting 225/1601: participant 1687315516
Extracting 250/1601: participant 177928631
Extracting 275/1601: participant 1833416906
Extracting 300/1601: participant 1919055552
Extracting 325/1601: participant 1998578790
  Missing/unusable FreeSurfer data for 2003542642: [Errno 2] No such file or directory: '/home/jovyan/shared/data/RBC/repos/PNC_FreeSurfer/freesurfer/sub-2003542642/sub-2003542642_region

,participant_id,p_factor,SurfArea__caudalanteriorcingulate,SurfArea__caudalmiddlefrontal,SurfArea__cuneus,SurfArea__entorhinal,SurfArea__fusiform
0,1000393599,0.589907,1728.0,4526.0,5168.0,1084.0,5579.0
1,1000881804,-0.655377,1527.0,3820.0,4278.0,714.0,6063.0
2,1001970838,-0.659061,1594.0,4186.0,3759.0,695.0,5310.0
3,100527940,-0.591516,2172.0,5053.0,4577.0,806.0,6061.0
4,1006151876,-0.377828,1389.0,3922.0,3488.0,713.0,4526.0


## Cross-Validate the Linear Regression

This evaluates the model on the known training participants only. The test participants' hidden `p_factor` values are not used.

In [5]:
X_train = model_data.loc[train_rows, usable_feature_cols]
y_train = model_data.loc[train_rows, "p_factor"]
X_test = model_data.loc[~train_rows, usable_feature_cols]

linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("linear_regression", LinearRegression()),
])

n_splits = min(5, len(y_train))
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_predictions = cross_val_predict(linear_model, X_train, y_train, cv=cv)

cv_metrics = pd.Series({
    "cross_validated_r2": r2_score(y_train, cv_predictions),
    "cross_validated_mae": mean_absolute_error(y_train, cv_predictions),
    "cross_validated_rmse": np.sqrt(mean_squared_error(y_train, cv_predictions)),
})

cv_metrics

/srv/conda/envs/notebook/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['SurfArea__frontalpole']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['SurfArea__frontalpole']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


cross_validated_r2     -0.018864
cross_validated_mae     0.778482
cross_validated_rmse    0.945370
dtype: float64

## Fit Final Model and Predict Test Participants

In [6]:
final_model = linear_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

predicted_test_data = test_data.copy()
predicted_test_data.loc[:, "p_factor"] = test_predictions

predicted_test_data.head()

,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,p_factor
0,2285120424,PNC,PNC1,PNC1,1,16.166667,Female,Black,not Hispanic or Latino,22.67,Left,9th Grade,Complete secondary,Complete secondary,-0.228837
1,1854364375,PNC,PNC1,PNC1,1,16.833333,Female,White,not Hispanic or Latino,28.19,Right,10th Grade,Complete secondary,Complete primary,-0.377414
2,1448081953,PNC,PNC1,PNC1,1,19.583333,Female,White,not Hispanic or Latino,25.10,Right,Some College,Complete tertiary,Complete tertiary,-0.429536
3,1342685465,PNC,PNC1,PNC1,1,17.166667,Female,Black,not Hispanic or Latino,31.93,Right,10th Grade,Complete primary,Complete primary,-0.195428
4,3289562482,PNC,PNC1,PNC1,1,21.166667,Female,White,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,-0.351402


## Inspect Largest Coefficients

Because the predictors are standardized inside the model pipeline, larger absolute coefficients indicate regions with stronger linear weight in this first-pass model.

In [7]:
coefficients = pd.Series(
    final_model.named_steps["linear_regression"].coef_,
    index=usable_feature_cols,
    name="standardized_coefficient",
)

top_coefficients = coefficients.reindex(
    coefficients.abs().sort_values(ascending=False).index
).head(20).to_frame()

top_coefficients

,standardized_coefficient
SurfArea__transversetemporal,-0.118216
SurfArea__precentral,0.113097
SurfArea__posteriorcingulate,-0.086020
SurfArea__isthmuscingulate,0.081639
SurfArea__superiortemporal,0.077233
SurfArea__caudalmiddlefrontal,-0.076854
SurfArea__lateralorbitofrontal,-0.075308
SurfArea__inferiorparietal,-0.072444
SurfArea__rostralanteriorcingulate,0.068902
SurfArea__inferiortemporal,-0.065854


## Save Predictions

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predicted_test_data.to_csv(OUTPUT_PATH, sep="\t", index=False)

print(f"Saved predictions to {OUTPUT_PATH}")
OUTPUT_PATH

Saved predictions to results/dkt_surfarea_linear.tsv


PosixPath('results/dkt_surfarea_linear.tsv')